# Bargaining over Cabinets: run on Google Colab

This notebook reproduces the experiments of the repository on Colab.
**A CPU runtime is enough**: the bottleneck is the Python simulation of the bargaining game,
not the (small) neural network, so a GPU gives little speed-up.

**Before you start**
1. *Runtime > Change runtime type*: CPU is fine (High-RAM if available).
2. The repository is private. Create a GitHub token with read access to it
   (GitHub > Settings > Developer settings > Fine-grained tokens > *Contents: Read-only* on this repo),
   then add it in Colab's **Secrets** panel (key icon on the left) under the name `GH_TOKEN`
   and enable *Notebook access*. The token is never printed.

In [ ]:
from google.colab import userdata
import os, subprocess
token = userdata.get('GH_TOKEN')
if not os.path.exists('cabinet-bargaining-marl'):
    subprocess.run(['git', 'clone', '-q',
                    f'https://{token}@github.com/bot2048/cabinet-bargaining-marl.git'], check=True)
%cd cabinet-bargaining-marl
!git log --oneline | head -5

## 1. Install and test (about 1 minute)

In [ ]:
!pip -q install -r requirements.txt
!python -m pytest -q

## 2. Quick demo (about 3 minutes)
Trains the shared PPO policy on the symmetric Baron-Ferejohn benchmark E0 for a few iterations
and prints the formateur premium, the gap to the analytic share and the failure rate.

In [ ]:
!python scripts/run_experiment.py --exp e0 --seed 0 --iters 60 --threads 2
import json; r = json.load(open('results/e0/seed0.json'))
print({k: round(v, 3) for k, v in r['overall'].items() if isinstance(v, float)})

## 3. Full experiments (optional)
`scripts/run_all.sh` runs every experiment cell. Colab has ~2 CPU cores, so run **one block at a time**;
each line below is one training run (roughly 10-20 minutes each on Colab).
Set `SEEDS` to the seeds you want to add (the repository already contains seeds 0-2).

In [ ]:
SEEDS = [3]
for s in SEEDS:
    !python scripts/run_experiment.py --exp e0 --seed {s} --threads 2
    !python scripts/run_experiment.py --exp e1 --seed {s} --threads 2
    for cfg in ['BASE', 'Estar']:
        !python scripts/run_experiment.py --exp mech --config {cfg} --seed {s} --threads 2

## 4. Figures and download

In [ ]:
!python scripts/make_figures.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('figures/*.png')):
    display(Image(f))
!zip -qr results_colab.zip results figures
from google.colab import files
files.download('results_colab.zip')